# Exploratory Data Analysis — EV Adoption Likelihood

**Dataset:** Global EV Adoption Behavior 2026

**Goal:** Understand the structure, distributions, missing values, and relationships before building a KNN classification model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add project root to path so we can import utils
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.preprocess import load_dataset, separate_features_target, detect_column_types

%matplotlib inline
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

---
## 1. Dataset Loading

In [ ]:
df = load_dataset()
print(f'Dataset shape: {df.shape}')
df.head()

---
## 2. Dataset Inspection

In [ ]:
print('Column names and data types:')
print(df.dtypes.to_string())

In [ ]:
print('Column summary:')
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'n_unique': df.nunique(),
    'n_missing': df.isnull().sum(),
    'pct_missing': (df.isnull().sum() / len(df) * 100).round(2),
})
summary

In [ ]:
print(f'Duplicate rows: {df.duplicated().sum()}')

---
## 3. Missing Values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(missing.index, missing.values, color='coral')
ax.bar_label(bars, labels=[f'{v} ({v/len(df)*100:.1f}%)' for v in missing.values])
ax.set_xlabel('Number of missing values')
ax.set_title('Missing Values per Column')
plt.tight_layout()
plt.show()

In [ ]:
# Check overlap of missing values
m1 = df['education_level'].isnull()
m2 = df['charging_station_accessibility'].isnull()
m3 = df['ev_knowledge_score'].isnull()

overlap = pd.DataFrame({
    'scenario': ['Only education_level', 'Only charging_station_accessibility', 'Only ev_knowledge_score',
                 'education_level + charging_station', 'education_level + ev_knowledge',
                 'charging_station + ev_knowledge', 'All three'],
    'count': [
        ((m1) & (~m2) & (~m3)).sum(),
        ((~m1) & (m2) & (~m3)).sum(),
        ((~m1) & (~m2) & (m3)).sum(),
        ((m1) & (m2) & (~m3)).sum(),
        ((m1) & (~m2) & (m3)).sum(),
        ((~m1) & (m2) & (m3)).sum(),
        ((m1) & (m2) & (m3)).sum(),
    ]
})
overlap

**Observation:** Missing values appear in only 3 columns and are largely non-overlapping (~500 each, ~1500 total rows affected).

---
## 4. Target Variable Distribution

In [ ]:
target_counts = df['ev_adoption_likelihood'].value_counts()
target_pct = df['ev_adoption_likelihood'].value_counts(normalize=True).mul(100).round(2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars1 = ax1.bar(target_counts.index, target_counts.values, color=colors)
ax1.bar_label(bars1)
ax1.set_title('Target Class Counts')
ax1.set_ylabel('Count')

bars2 = ax2.bar(target_pct.index, target_pct.values, color=colors)
ax2.bar_label(bars2, suffix='%')
ax2.set_title('Target Class Proportions')
ax2.set_ylabel('Percentage (%)')

plt.tight_layout()
plt.show()

print('Target distribution:')
print(target_pct.to_string())

**Observation:** The target is imbalanced — "High" is the majority class (~59%), followed by "Medium" (~24%) and "Low" (~17%).

---
## 5. Numerical Features — Descriptive Statistics

In [ ]:
X, y = separate_features_target(df)
col_types = detect_column_types(X)

print('Numerical columns:')
print(col_types['numerical'])
print()
print('Binary columns:')
print(col_types['binary'])
print()
print('Ordinal categorical columns:')
print(col_types['ordinal'])
print()
print('Nominal categorical columns:')
print(col_types['nominal'])

In [ ]:
df[col_types['numerical']].describe().T

In [ ]:
# Check for anomalies in numerical features
print('Columns with negative values (where unexpected):')
for col in col_types['numerical']:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f'  {col}: {neg_count} negative values (min={df[col].min():.2f})')

**Observation:** `fuel_expense_per_month` has some negative values which are data errors. These need to be handled in preprocessing.

---
## 6. Distribution Plots (Numerical Features)

In [ ]:
num_cols = col_types['numerical']
n_cols = 5
n_rows = (len(num_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

---
## 7. Categorical Features — Value Counts

In [ ]:
cat_cols = col_types['ordinal'] + col_types['nominal']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(15, 4))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values, color='steelblue')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())

---
## 8. Correlation Heatmap (Numerical Features)

In [ ]:
plt.figure(figsize=(18, 14))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            square=True, cbar_kws={'shrink': 0.8}, linewidths=0.5)
plt.title('Correlation Heatmap (Numerical Features)')
plt.tight_layout()
plt.show()

---
## 9. Target vs. Key Numerical Features

In [ ]:
key_features = ['age', 'annual_income', 'daily_commute_km', 'fuel_expense_per_month',
                'environmental_awareness_score', 'technology_affinity_score',
                'range_anxiety_score', 'ev_knowledge_score']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

target_order = ['Low', 'Medium', 'High']

for i, col in enumerate(key_features):
    for j, level in enumerate(target_order):
        subset = df[df['ev_adoption_likelihood'] == level][col].dropna()
        axes[i].hist(subset, bins=40, alpha=0.5, label=level, density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 10. Summary Observations

In [ ]:
print('=' * 70)
print('EDA SUMMARY')
print('=' * 70)
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print(f'Numerical features: {len(col_types["numerical"])}')
print(f'Binary features: {len(col_types["binary"])}')
print(f'Categorical ordinal features: {len(col_types["ordinal"])}')
print(f'Categorical nominal features: {len(col_types["nominal"])}')
print()
print('Missing values: 3 columns with ~500 missing each')
print('  - education_level')
print('  - charging_station_accessibility')
print('  - ev_knowledge_score')
print()
print('Data quality issues:')
print('  - Negative fuel_expense_per_month values (~271 rows)')
print()
print('Target imbalance:')
print(f'  High:   {target_pct["High"]}%')
print(f'  Medium: {target_pct["Medium"]}%')
print(f'  Low:    {target_pct["Low"]}%')
print()
print('Encoding decisions:')
print('  - education_level  -> Label Encoding (ordinal)')
print('  - city_type        -> OneHot Encoding (nominal)')
print('  - current_vehicle_type -> OneHot Encoding (nominal)')
print('  - home_charging_available, previous_ev_experience -> kept as-is (binary)')
print()
print('Scaling: StandardScaler on all numerical features')
print('Split: 80/20 stratified by target')